In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 1 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================
#
# FINAL WEEK STRATEGY
#
# Function 1 remains a special case:
#
# - maximise the ORIGINAL objective
# - use only positive linear scaling
# - do NOT use a log transform
# - treat the GP mainly as a LOCAL RANKING model
#
# Week 11:
# [0.698187, 0.734173] -> 6.59126e-12  (current best)
#
# Week 12:
# [0.721092, 0.734173] -> 1.56082e-14
#
# The Week 12 move increased x1 while keeping x2 fixed and
# performed substantially worse.
#
# Since this is the FINAL query:
# - prioritise exploitation
# - stay close to the proven incumbent
# - use highest mean + low-beta UCB as primary evidence
# - use EI only as a diagnostic
# - explicitly examine the x1 direction
# - do not expand the search simply because uncertainty is high
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function1/initial_inputs.npy"
)

Y = np.load(
    "function1/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 12 calibration check
# ------------------------------------------------------------
#
# Week 12 selected point came from the x1 axis profile.
#
# Previous prediction was approximately:
#
# mean = 3.2458414e-07
# std  = 6.2347774e-07
#
# Actual:
# 1.56082198768421e-14
#
# The standardised residual is still moderate, but the raw
# predicted level remains far too large. Therefore the GP is
# still used for ranking rather than absolute prediction.
# ------------------------------------------------------------

week12_x = np.array([
    0.721092,
    0.734173
])

week12_actual = (
    1.56082198768421e-14
)

week12_pred_mean = (
    3.245841435839448e-07
)

week12_pred_std = (
    6.234777359151375e-07
)

week12_error = (
    week12_actual
    - week12_pred_mean
)

week12_z_error = (
    week12_error
    / week12_pred_std
)

print("\n================================")
print("WEEK 12 CALIBRATION CHECK")
print("================================")

print(
    "Predicted raw mean:",
    week12_pred_mean
)

print(
    "Predicted raw std :",
    week12_pred_std
)

print(
    "Actual            :",
    week12_actual
)

print(
    "\nPrediction error:",
    week12_error
)

print(
    "\nError / predicted std:",
    week12_z_error
)


# ------------------------------------------------------------
# 3. Recent directional evidence
# ------------------------------------------------------------

week10_x = np.array([
    0.757302,
    0.735569
])

week10_y = (
    -1.1724650608954473e-20
)

week11_x = np.array([
    0.698187,
    0.734173
])

week11_y = (
    6.591260182136635e-12
)

print("\n================================")
print("RECENT LOCAL EVIDENCE")
print("================================")

print("\nWeek 10:")
print(
    week10_x,
    "->",
    week10_y
)

print("\nWeek 11 / incumbent:")
print(
    week11_x,
    "->",
    week11_y
)

print("\nWeek 12:")
print(
    week12_x,
    "->",
    week12_actual
)

print(
    "\nWeek 12 x1 displacement from incumbent:"
)

print(
    week12_x[0]
    - best_x[0]
)


# ------------------------------------------------------------
# 4. Positive linear scaling
# ------------------------------------------------------------
#
# This preserves the exact ordering and argmax.
# ------------------------------------------------------------

y_scale = np.max(
    np.abs(Y)
)

Y_scaled = (
    Y / y_scale
)

best_y_scaled = (
    best_y / y_scale
)

print("\n================================")
print("POSITIVE LINEAR SCALING")
print("================================")

print(
    "y_scale:",
    y_scale
)

print(
    "Original argmax:",
    np.argmax(Y)
)

print(
    "Scaled argmax:",
    np.argmax(Y_scaled)
)

print(
    "Ordering preserved:",
    np.argmax(Y)
    == np.argmax(Y_scaled)
)


# ------------------------------------------------------------
# 5. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(
            0.005,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(
            1e-10,
            1e-2
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y_scaled
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 6. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 7. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Final week:
# keep the search local around the proven incumbent.

empirical_cap = min(
    1.25 * nearest_distance,
    0.035
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 8. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.20 * lengthscales,
    0.006,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 13 TRUST REGION")
print("================================")

print(
    "Centre:"
)
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 9. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        450000,
        2
    )
)

# Portal precision before duplicate checking
candidates = np.round(
    candidates,
    6
)

tree = cKDTree(
    X
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 10. GP predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled
    * y_scale
)

sigma_raw = (
    sigma_scaled
    * y_scale
)


# ------------------------------------------------------------
# 11. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "raw mean =",
    mu_raw[ei_idx]
)

print(
    "raw std =",
    sigma_raw[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 12. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu_scaled
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "scaled mean =",
    mu_scaled[mean_idx]
)

print(
    "scaled std =",
    sigma_scaled[mean_idx]
)

print(
    "raw mean =",
    mu_raw[mean_idx]
)

print(
    "raw std =",
    sigma_raw[mean_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 13. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta
        * sigma_scaled
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "raw mean =",
        mu_raw[idx]
    )

    print(
        "raw std =",
        sigma_raw[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 14. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta
        * sigma_scaled
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 15. x1 profile through the incumbent
# ------------------------------------------------------------
#
# This is especially important in the final week.
#
# Week 12 tested an upward x1 move and performed worse.
# Search systematically on both sides of the incumbent while
# holding x2 at the successful Week 11 value.
# ------------------------------------------------------------

x1_values = np.linspace(
    lower[0],
    upper[0],
    60000
)

x1_profile = np.column_stack([
    x1_values,
    np.full_like(
        x1_values,
        best_x[1]
    )
])

x1_profile = np.round(
    x1_profile,
    6
)

profile_distance, _ = tree.query(
    x1_profile,
    k=1
)

x1_valid = x1_profile[
    profile_distance > 0.01
]

print("\n================================")
print("x1 PROFILE THROUGH INCUMBENT")
print("================================")

if len(x1_valid) > 0:

    x1_mu_scaled, x1_sigma_scaled = gp.predict(
        x1_valid,
        return_std=True
    )

    x1_idx = np.argmax(
        x1_mu_scaled
    )

    print(
        "Best x1 profile candidate =",
        x1_valid[x1_idx]
    )

    print(
        "raw mean =",
        x1_mu_scaled[x1_idx]
        * y_scale
    )

    print(
        "raw std =",
        x1_sigma_scaled[x1_idx]
        * y_scale
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            x1_valid[x1_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible x1-profile points."
    )


# ------------------------------------------------------------
# 16. x2 profile through incumbent
# ------------------------------------------------------------

x2_values = np.linspace(
    lower[1],
    upper[1],
    40000
)

x2_profile = np.column_stack([
    np.full_like(
        x2_values,
        best_x[0]
    ),
    x2_values
])

x2_profile = np.round(
    x2_profile,
    6
)

profile_distance, _ = tree.query(
    x2_profile,
    k=1
)

x2_valid = x2_profile[
    profile_distance > 0.01
]

print("\n================================")
print("x2 PROFILE THROUGH INCUMBENT")
print("================================")

if len(x2_valid) > 0:

    x2_mu_scaled, x2_sigma_scaled = gp.predict(
        x2_valid,
        return_std=True
    )

    x2_idx = np.argmax(
        x2_mu_scaled
    )

    print(
        "Best x2 profile candidate =",
        x2_valid[x2_idx]
    )

    print(
        "raw mean =",
        x2_mu_scaled[x2_idx]
        * y_scale
    )

    print(
        "raw std =",
        x2_sigma_scaled[x2_idx]
        * y_scale
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            x2_valid[x2_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible x2-profile points."
    )


# ------------------------------------------------------------
# 17. Symmetric response to Week 12 move
# ------------------------------------------------------------
#
# Week 12 moved:
#
# incumbent -> higher x1
#
# and the objective became worse.
#
# The mirror point tests the same step length in the opposite
# x1 direction. This is not automatically selected; it is only
# a systematic directional diagnostic.
# ------------------------------------------------------------

week12_step = (
    week12_x
    - best_x
)

mirror_candidate = (
    best_x
    - week12_step
)

mirror_candidate = np.clip(
    mirror_candidate,
    0.0,
    1.0
)

mirror_candidate = np.round(
    mirror_candidate,
    6
)

mirror_dist, _ = tree.query(
    mirror_candidate.reshape(1, -1),
    k=1
)

print("\n================================")
print("WEEK 12 MIRROR DIAGNOSTIC")
print("================================")

print(
    "Mirror candidate =",
    mirror_candidate
)

print(
    "Nearest observed distance =",
    mirror_dist[0]
)

if mirror_dist[0] > 0.01:

    mirror_mu_scaled, mirror_sigma_scaled = gp.predict(
        mirror_candidate.reshape(1, -1),
        return_std=True
    )

    print(
        "raw mean =",
        mirror_mu_scaled[0]
        * y_scale
    )

    print(
        "raw std =",
        mirror_sigma_scaled[0]
        * y_scale
    )

else:

    print(
        "Mirror candidate is too close to an existing observation."
    )

DATA
X shape: (22, 2)
Y shape: (22,)

Current best:
[0.698187 0.734173] -> 6.591260182136635e-12

Y range:
min = -0.0036060626443634764
max = 6.591260182136635e-12
std = 0.0007511388217777337

WEEK 12 CALIBRATION CHECK
Predicted raw mean: 3.245841435839448e-07
Predicted raw std : 6.234777359151375e-07
Actual            : 1.56082198768421e-14

Prediction error: -3.245841279757249e-07

Error / predicted std: -0.5206025961765931

RECENT LOCAL EVIDENCE

Week 10:
[0.757302 0.735569] -> -1.1724650608954473e-20

Week 11 / incumbent:
[0.698187 0.734173] -> 6.591260182136635e-12

Week 12:
[0.721092 0.734173] -> 1.56082198768421e-14

Week 12 x1 displacement from incumbent:
0.022904999999999953

POSITIVE LINEAR SCALING
y_scale: 0.0036060626443634764
Original argmax: 20
Scaled argmax: 20
Ordering preserved: True


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.254**2 * Matern(length_scale=[2, 0.0394], nu=2.5) + WhiteKernel(noise_level=1e-10)

ARD lengthscales:
[2.        0.0394229]

Normalised inverse-lengthscale sensitivity:
[0.01933042 0.98066958]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.022904999999999953

Empirical cap: 0.02863124999999994

WEEK 13 TRUST REGION
Centre:
[0.698187 0.734173]

Half-widths:
[0.02863125 0.00788458]

Lower:
[0.66955575 0.72628842]

Upper:
[0.72681825 0.74205758]

Candidates after portal rounding + duplicate filtering:
196730

PRIMARY EI - DIAGNOSTIC ONLY
candidate = [0.670721 0.742057]
raw mean = -2.751753735950586e-06
raw std = 1.809983994178777e-05
EI = 0.0016439539506109388
distance from incumbent = 0.028575139754688856

HIGHEST PREDICTED MEAN
candidate = [0.669632 0.733764]
scaled mean = 4.831310663178101e-05
scaled std = 0.0002209854429957083
raw mean = 1.7422008905801483e-07
raw std = 7.968873509349381e-07
distance from incumbent = 0.02855792895151887

UCB DIAGNOSTICS

be

In [2]:
# ============================================================
# FINAL FUNCTION 1 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - Week 12 increased x1 and performed worse than the incumbent.
# - Updated highest posterior mean moves x1 downward.
# - beta=0.05 and beta=0.10 UCB agree on the same direction.
# - The dedicated x1 profile also reaches its best value at
#   the lower side of the trust region.
# - EI / larger UCB values move toward uncertain x2 regions
#   and have negative posterior means, so they are rejected.
#
# x2 is kept exactly at the successful incumbent value because
# the GP remains much more locally sensitive in x2.
#
# The GP's absolute predicted objective remains poorly scaled,
# so this decision uses its LOCAL RANKING rather than treating
# the predicted magnitude as trustworthy.
#
# We do NOT expand the trust region further in the final week.
# ============================================================

week13_candidate = np.array([
    0.669556,
    0.734173
])

print("================================")
print("FUNCTION 1 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)

# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu_scaled, final_sigma_scaled = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

final_mu_raw = (
    final_mu_scaled[0]
    * y_scale
)

final_sigma_raw = (
    final_sigma_scaled[0]
    * y_scale
)

print("\nPredicted raw mean:")
print(final_mu_raw)

print("\nPredicted raw std:")
print(final_sigma_raw)


# ------------------------------------------------------------
# Distance from incumbent
# ------------------------------------------------------------

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week13_candidate
        - best_x
    )
)


# ------------------------------------------------------------
# Check against all previous observations
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(
    nearest_dist[0]
)

print("\nNearest observed point:")
print(
    X[nearest_idx[0]]
)

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.01
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 1 - WEEK 13 FINAL

Candidate:
[0.669556 0.734173]

Predicted raw mean:
1.425498431019075e-07

Predicted raw std:
6.687782653445415e-07

Distance from current best:
0.028630999999999962

Nearest observed-point distance:
0.028630999999999962

Nearest observed point:
[0.698187 0.734173]

Duplicate rule satisfied:
True

Portal format:
0.669556-0.734173
